# UAV FP — Optimización del ala volante (panel de control)

La lógica estable (geometría, función objetivo, backend de cruzamiento, loop evolutivo) vive en el paquete **`optimizacion_ala/`**. Este notebook es el **panel de control**.

**Modelo aerodinámico: `AeroBuildup` (AeroSandbox).** Calcula las **dos** contribuciones de resistencia y las suma:

$$D_{total} = D_{inducida} + D_{perfil}$$

- **Inducida** — la que produce la sustentación finita. Es la única que daba el VLM.
- **De perfil** — viscosa (fricción de piel + presión), estimada con **NeuralFoil** por estación de envergadura a la Re local.

*Hasta el 2026-08-26 el objetivo usaba VLM invíscido, que solo calculaba la inducida. Como la inducida ∝ CL², el L/D tendía a infinito cuando el CL tendía a cero, y el GA aprendió a explotarlo: llegó a un "ganador" con L/D = 551 y CL = 0.022 — un ala que no sustentaba nada. Ese agujero está cerrado: ver la sección 1.*

**Backend de generación de hijos: GA numérico** (BLX-alfa + mutación gaussiana), sin IA — no necesita `claude login` ni `ANTHROPIC_API_KEY`.

**Parametrización: 17 parámetros, envergadura FIJA** (`HALF_SPAN` = 1.1 m, o sea 2.2 m de punta a punta; se edita en `geometria.py`). Borde de ataque curvo, cuerda y torsión en 4 estaciones, perfil reflejado custom por 5 puntos de curvatura.

**Cómo correrlo:** secciones 1 y 2 para configurar y sembrar; después **3** (una generación por vez) o **4** (automático hasta que deje de mejorar). Las secciones 5-8 son para mirar el resultado.

**El objetivo pesa performance y estabilidad:** `score = 1.0·(L/D)/30 + 0.5·p_estab(SM)`, con el margen estático apuntado a la banda **5-10%** (ver sección 1). Los candidatos con margen estático negativo se descartan.

**Pendiente (Research Note 11):** falta **peso estructural**, **margen de entrada en pérdida**, y evaluar a sustentación fija en vez de alpha fijo. También falta exigir equilibrio (trim) en el score — `calcular_aero()` ya devuelve `Cm0` y `CL_trim`, pero todavía no entran a la cuenta.

In [ ]:
# Autorecarga: si se editan los .py del paquete optimizacion_ala/ mientras el
# notebook esta abierto, Jupyter NO los recarga solo -- se queda con la version
# que importo la primera vez. Estas dos lineas hacen que cada celda que corras
# use la version actual del disco, sin tener que reiniciar el kernel.
# (Si aparece un AttributeError del tipo "module has no attribute PESO_...",
#  es exactamente ese problema: reinicia el kernel una vez y listo.)
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# El paquete optimizacion_ala/ vive al lado de este notebook.
sys.path.insert(0, str(Path.cwd()))

import optimizacion_ala as opt
import optimizacion_ala.objetivo as obj
from optimizacion_ala.perfiles import generar_hijos_con_catalogo as generar_hijos
# ^ backend activo: GA numerico + wildcards sembrados desde perfiles conocidos.
#   Para el backend anterior (wildcards uniformes):
#   from optimizacion_ala.ga_numerico import generar_hijos

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json


## 1. Configuración: función objetivo y criterio de parada

| Función | Qué calcula | Cuándo usarla |
|---|---|---|
| `evaluar_candidato` | **L/D + estabilidad**, pesados | **El default** |
| `evaluar_candidato_invisido` | L/D solo inducida (VLM) — el objetivo viejo, roto | Solo para mostrar el contraste |
| `evaluar_candidato_toy` | Fórmula algebraica, sin aerodinámica | Probar el loop sin esperar al solver |

### El objetivo

$$\text{score} \;=\; w_{L/D}\,\frac{L/D}{(L/D)_{ref}} \;+\; w_{estab}\;p_{estab}(SM)$$

`PESO_LD = 1.0`, `PESO_ESTABILIDAD = 0.5`, `LD_REFERENCIA = 30`. Todo editable arriba de `objetivo.py`.

### Estabilidad: banda de margen estático 5–10%

Un ala volante usa márgenes **mucho más chicos** que un avión convencional (10-20%), porque no tiene cola que le dé amortiguamiento en cabeceo. La literatura tailless coincide en **~5% como punto de partida, rango 0-10%**, y por debajo de ~2% se vuelve incontrolable ([XFLR5](https://sourceforge.net/p/xflr5/discussion/679396/thread/1d1bff10/), [rcplanes.online](https://rcplanes.online/cg_wing.htm)).

Apuntamos al **extremo estable, [5%, 10%]**: es una plataforma de monitoreo que debe volar estable con viento y eventualmente llevar piloto automático. El puntaje es una **meseta** (vale 1 en toda la banda), así el optimizador entra a la banda y desde ahí dedica todo su esfuerzo al L/D. **Dos restricciones de validez** (descartan el candidato, no le bajan el puntaje):

1. **`SM ≤ 0` → descartado.** Un ala volante inestable no vuela sin control activo.
2. **`CL de equilibrio` fuera de [0.10, 1.00] → descartado.** Ser estable no alcanza: el ala también tiene que poder **equilibrarse** a sustentación positiva y útil. Como `CL_trim = Cm0 / SM` y ya exigimos `SM > 0`, esto equivale a pedir **`Cm0 > 0`** — que es lo que consiguen el reflejo del perfil (`z4`, `z5`) y el washout.

   *Por qué se agregó:* con estabilidad en el score pero sin esta restricción, el optimizador entregó un "ganador" con SM = 8.36% (perfecto), L/D = 31.4 (excelente)… y **Cm0 = −0.067, CL de equilibrio = −0.80**. Un avión estable, eficiente, y que solo se equilibra volando invertido. Es la misma clase de agujero que el L/D = 551 del objetivo invíscido: el optimizador encuentra lo que no le pediste explícitamente. Con la restricción puesta, el score baja de 1.546 a 1.431 — esa diferencia era diseño inválido.

### Parametrización: 30 parámetros

| Grupo | Parámetros | Unidad |
|---|---|---|
| Flecha global | `sweep_deg` | grados |
| Cuerda (6 estaciones) | `root_chord`, `chord_1..5` | **metros** |
| Borde de ataque curvo | `le_offset_1..5` | **fracción de la semi-envergadura** |
| Torsión (6 estaciones) | `twist_0..5` | grados |
| Perfil (línea media) | `z1..z8` | **fracción de la cuerda local** |
| Winglets | `winglet_height`, `winglet_cant_deg`, `winglet_taper`, `winglet_sweep_deg` | m, grados, ratio, grados |

### Tu pregunta: ¿los puntos de control son fracción del half-span o valores fijos?

**Depende del grupo, y la mezcla es deliberada:**

- **Estaciones de envergadura y `le_offset_*` → fracción de `HALF_SPAN`.** Si mañana cambiás la envergadura, la *forma* del ala se mantiene y solo cambia la escala. (Los `le_offset` estaban en metros absolutos hasta esta versión; los pasé a fracción justamente por esto.)
- **Cuerdas → metros absolutos.** Sus límites son restricciones de **fabricación** (abajo de ~80 mm no entra el herraje del elevón). Eso es un número absoluto, no proporcional a nada.
- **`z1..z8` → fracción de la cuerda local.** Así es como se define un perfil: el mismo perfil escala con la cuerda de cada estación.

### El "pozo" en el medio: qué era y cómo se arregló

No era el ala, era el **perfil**, y tenía dos causas que se sumaban:

1. **Los límites de curvatura eran absurdos.** `z1..z5` iban de 0 a **0.20**, o sea hasta 20% de curvatura sobre la cuerda. Un perfil real anda en **2-6%**. Con `z` en el medio de sus límites (0.10), la línea media saltaba de 0 en el borde de ataque a 0.10 en el primer 10% de la cuerda: una pared casi vertical seguida de una meseta. Eso no es un perfil, es un gancho — y como la cuerda de raíz es la más grande, el gancho se veía enorme en el centro. **Ese era el pozo.**
2. **La interpolación era lineal** (`np.interp`), o sea una poligonal con un quiebre en cada punto de control. Nada de curvas suaves.

Qué cambió:

- **Límites físicos** para la curvatura, y ahora se permiten valores **negativos**. Eso último importa mucho: un perfil reflejado —el que necesita un ala volante sin cola— tiene curvatura positiva adelante y **negativa** cerca del borde de fuga. Con el rango viejo, que arrancaba en 0, la forma en S ni siquiera era representable.
- **Interpolación PCHIP** (spline cúbica monótona) en vez de lineal, para la curvatura y para las distribuciones a lo largo de la envergadura. PCHIP y no una cúbica común porque PCHIP **no se pasa**: entre dos puntos de control no inventa jorobas que nadie pidió.
- **Puntos de control más densos cerca de los bordes** (espaciado tipo coseno: 0.02, 0.06, 0.12 … 0.88, 0.96), que es lo que pediste — el cambio es gradual donde la curvatura manda, y ralo en el medio, que es suave por naturaleza. Pasaron de 5 a 8.
- **El ala se loftea a través de 11 secciones interpoladas**, no las 6 de control. Las 6 siguen siendo las variables de diseño (el optimizador no ve más parámetros), pero la superficie que recibe el solver es suave en vez de facetada. El 11 sale de un estudio de convergencia: de ahí en adelante el L/D no se mueve (menos de 0.03% hasta 21 secciones) y el costo sigue subiendo lineal. Para **graficar** se piden 80 sin tocar nada.

### Winglets y el punto de operación — leer antes de sacar conclusiones

El winglet baja la resistencia **inducida** a costa de más superficie mojada (más resistencia de **perfil**). Cuál gana depende de qué fracción del total es inducida:

| α | CL | inducida / total | ¿conviene? |
|---|---|---|---|
| 4° (**nuestro OP_POINT**) | 0.14 | **10.9%** | **no** |
| 8° | 0.47 | 31.9% | **sí** |
| 10° | 0.61 | 29.3% | **sí** |

Con `OP_POINT` en α = 4°, esperá winglets chicos o nulos. No es un bug: es el modelo diciendo que a ese punto de vuelo no pagan.

### Criterio de parada: por qué `PACIENCIA = 8`

La corrida anterior frenaba a las 9 generaciones. Con 27 parámetros el espacio es más grande y cortar tan temprano deja mucho sin explorar. Medido, 2 semillas por configuración:

| config | generaciones | scores | media |
|---|---|---|---|
| paciencia 3 (lo que había) | 12, 9 | 1.454, 1.397 | 1.425 |
| **paciencia 8** | 45, 26 | 1.557, 1.477 | **1.517** |
| paciencia 8 + sigma adaptativo | 11, 8 | 1.32, 1.298 | 1.309 |

**Subir la paciencia mejora** (+6.5%, en las dos semillas). En cambio **el sigma adaptativo empeora**: era mi hipótesis para escapar de la convergencia prematura (agrandar la mutación al estancarse), y resultó contraproducente — las mutaciones grandes producen hijos peores, la racha sin mejora crece más rápido, y el loop corta *antes*. Lo probé y lo descarté; no está en el código.

Con paciencia 8, una semilla llegó al tope de 45 generaciones **todavía mejorando**, así que `MAX_GENERACIONES = 80`. Una corrida completa tarda unos pocos minutos.

---

### Lo que todavía queda flojo

El diseño final se **evalúa** a α = 4° (CL ≈ 0.34) pero se **equilibra** a CL ≈ 0.16. O sea, los números de L/D corresponden a una condición en la que el avión no está trimado. Cerrar eso es el pendiente grande de la Nota 11: **evaluar en la condición de equilibrio** (buscar el α donde Cm = 0) en vez de a α fijo. Es lo que también le daría chance real a los winglets, porque movería el punto de operación hacia CL más altos, donde la resistencia inducida pesa más.

In [ ]:
# --- Funcion objetivo ---
OBJETIVO_FN = opt.evaluar_mision               # MISION: crucero 60 km/h + 3 kg, con estructura y rafagas
# OBJETIVO_FN = opt.evaluar_candidato          # anterior: L/D + estabilidad, masa declarada
# OBJETIVO_FN = opt.evaluar_candidato_toy      # rapidisima, sin aerodinamica
# OBJETIVO_FN = opt.evaluar_candidato_invisido # el objetivo VIEJO y roto, solo para comparar

# --- Criterio de parada ---
PACIENCIA = 8
MAX_GENERACIONES = 40

# --- Cuantos hijos por generacion ---
N_CROSSBRED, N_WILDCARD = 7, 3

import optimizacion_ala.mision as mis
print("Objetivo:", OBJETIVO_FN.__name__)
if OBJETIVO_FN is opt.evaluar_mision:
    print(f"  MISION: crucero {3.6*mis.V_CRUCERO:.0f} km/h con {mis.MASA_PAYLOAD:.1f} kg de carga util")
    print(f"  masa CALCULADA por el modelo estructural (no declarada)")
    print(f"  rafaga vertical de diseno: {mis.RAFAGA_VERTICAL:.1f} m/s")
    print(f"  pesos: eficiencia {mis.PESO_EFICIENCIA} | masa {mis.PESO_MASA} | estabilidad {mis.PESO_ESTABILIDAD}")
    print(f"         rafaga {mis.PESO_RAFAGA} | margen perdida {mis.PESO_MARGEN} | trim {mis.PESO_TRIM}")
else:
    print(f"  score = {obj.PESO_LD} * (L/D)/{obj.LD_REFERENCIA:.0f}  +  {obj.PESO_ESTABILIDAD} * p_estab(SM)")
print(f"Banda de margen estatico: [{100*mis.SM_MIN:.0f}%, {100*mis.SM_MAX:.0f}%]")
print(f"CG al {100*opt.geometria.CG_FRAC_MAC:.0f}% de la MAC")
print(f"Envergadura FIJA: {2*opt.HALF_SPAN:.2f} m | {len(opt.BOUNDS)} parametros de diseno")
print(f"Parar tras {PACIENCIA} generaciones sin mejora, o al llegar a {MAX_GENERACIONES}.")


## 2. Población semilla — el optimizador ELIGE perfiles, no los inventa

**Qué cambió y por qué (2026-08-28).** Hasta acá el perfil era una variable continua: 8 puntos de curvatura (`z1..z8`) más un espesor, deformables libremente. Ese esquema **se degeneraba**. El motivo no es un bug sino algo estructural del método: NeuralFoil es una red entrenada sobre perfiles *reales*. Cuando se la evalúa en formas que no se parecen a nada de su set de entrenamiento —y con 9 grados de libertad libres se llega ahí rápido— sigue devolviendo un número, pero ese número es una **extrapolación**, no una predicción. Y como el optimizador busca máximos, va derecho a esas zonas: son justamente las que prometen más rendimiento del que existe.

La solución no es restringir más la forma, es **sacarle la posibilidad de inventar perfiles**. Ahora elige de un catálogo de perfiles reales de la base UIUC, en **tres anclajes**: raíz, medio y punta — y decide también **dónde** va el del medio.

**El catálogo está ordenado por Cm0 medido**, y eso no es cosmético: el perfil elegido es una variable discreta (un índice), pero BLX-alfa interpola valores continuos. Interpolar entre el índice 2 y el 9 solo tiene sentido si índices vecinos son diseños vecinos. Ordenado por Cm0, moverse un índice es moverse un poco sobre el compromiso central del ala volante:

| idx | perfil | Cm0 | espesor | L/D máx | CLmáx |
|---|---|---|---|---|---|
| 0 | naca2412 | −0.0586 | 12.0% | 82.5 | 1.324 |
| 1 | rg15 | −0.0558 | 8.9% | **83.7** | 1.224 |
| 2 | mh60 | −0.0032 | 10.1% | 74.7 | 1.221 |
| 3 | s5010 | +0.0034 | 9.8% | 78.6 | 1.284 |
| 4 | eh3012 | +0.0039 | 12.0% | 83.2 | 1.206 |
| 5 | eh2010 | +0.0061 | 10.1% | 79.2 | 1.134 |
| 6 | eh2012 | +0.0095 | 12.0% | 78.9 | 1.151 |
| 7 | clarkys | +0.0174 | 11.7% | 77.1 | 1.271 |
| 8 | e186 | +0.0176 | 10.3% | 73.1 | 1.042 |
| 9 | mh80 | +0.0187 | 12.7% | 73.3 | **1.527** |
| 10 | e340 | +0.0315 | 13.7% | 75.5 | 1.380 |
| 11 | mh78 | **+0.0489** | 14.5% | 66.2 | 1.451 |

**Más reflejo = más Cm0 = menos L/D.** Del naca2412 al mh78 se ganan 0.107 de Cm0 y se pierden 16 puntos de L/D. Los cuatro primeros (Cm0 ≤ 0) no equilibran un ala sin cola por sí solos; se dejan disponibles porque con mucho washout, o usados **solo en la punta**, son estrategias legítimas.

**El espesor dejó de ser variable**: sale del perfil elegido. Declararlo aparte era inconsistente — se le pedía a NeuralFoil un perfil de una forma y al modelo estructural una altura de larguero de otra.

**En su lugar entró el DIEDRO** (`elev_1..elev_5`), que no existía: el ala era plana por omisión, y nadie había decidido que el diedro valiera cero. Es la palanca principal de estabilidad lateral.

Las semillas arrancan con **perfil único en las tres zonas**: así, tener perfiles distintos por zona se tiene que *ganar* contra la alternativa simple en vez de venir impuesto.

In [ ]:
# Dos planforms distintos (uno CON winglet, otro sin) x los perfiles reflejados.
# Sembrar variado importa: BLX-alfa entre dos padres IGUALES en una dimension
# devuelve siempre ese mismo valor, asi que una dimension donde todas las
# semillas coinciden queda sin explorar. Le paso a los winglets.
# Por eso los dos planforms difieren tambien en winglet_radio (0.35 vs 0) y en
# el diedro (elev_*): si todos valieran lo mismo, no se explorarian.
PLANFORMS = [
    dict(sweep_deg=26.0, root_chord=0.42, chord_1=0.37, chord_2=0.31,
         chord_3=0.25, chord_4=0.19, chord_5=0.13,
         le_offset_1=0.005, le_offset_2=0.012, le_offset_3=0.022,
         le_offset_4=0.035, le_offset_5=0.050,
         twist_0=2.0, twist_1=0.5, twist_2=-0.5, twist_3=-1.5, twist_4=-2.5, twist_5=-4.0,
         elev_1=0.004, elev_2=0.010, elev_3=0.018, elev_4=0.028, elev_5=0.040,  # ~2 grados de diedro
         perfil_pos_medio=0.45,
         winglet_height=0.12, winglet_cant_deg=25.0, winglet_taper=0.6, winglet_sweep_deg=28.0,
         winglet_radio=0.35),
    dict(sweep_deg=32.0, root_chord=0.36, chord_1=0.32, chord_2=0.27,
         chord_3=0.22, chord_4=0.16, chord_5=0.10,
         le_offset_1=0.010, le_offset_2=0.020, le_offset_3=0.035,
         le_offset_4=0.050, le_offset_5=0.070,
         twist_0=1.0, twist_1=0.0, twist_2=-1.0, twist_3=-2.0, twist_4=-3.5, twist_5=-6.0,
         elev_1=0.000, elev_2=0.000, elev_3=0.000, elev_4=0.000, elev_5=0.000,  # ala plana
         perfil_pos_medio=0.60,
         winglet_height=0.0, winglet_cant_deg=20.0, winglet_taper=0.6, winglet_sweep_deg=25.0,
         winglet_radio=0.0),
]

seed_population = opt.poblacion_semilla(PLANFORMS)   # 2 planforms x 9 perfiles reflejados
estado = opt.iniciar_evolucion(seed_population, OBJETIVO_FN)

print(f"Poblacion semilla: {len(seed_population)} disenos "
      f"({len(PLANFORMS)} planforms x {len(opt.CATALOGO_REFLEJADOS)} perfiles)")
print("Padres iniciales (los 2 mejores):")
for c in estado.parents:
    perf = opt.perfiles_del_diseno(c.params())
    print(f"  score={c.score:.4f}  flecha={c.sweep_deg:.1f} deg  "
          f"perfiles={perf['raiz']}/{perf['medio']}/{perf['punta']}  "
          f"winglet={1000*c.winglet_height:.0f} mm")

## 3. Modo paso a paso — una generación por vez

**Volvé a correr esta celda tantas veces como quieras.** Útil para inspeccionar qué está proponiendo el GA. Si preferís dejarlo correr solo, saltá a la sección 4.

In [ ]:
if opt.deberia_parar(estado, paciencia=PACIENCIA, max_generaciones=MAX_GENERACIONES):
    print(f"Ya se cumplio el criterio de parada en la generacion {estado.generation} "
          f"(racha sin mejora: {estado.non_improving_streak}/{PACIENCIA}).")
else:
    estado, hijos = opt.correr_generacion(
        estado, generar_hijos, OBJETIVO_FN,
        n_crossbred=N_CROSSBRED, n_wildcard=N_WILDCARD,
    )
    opt.guardar_checkpoint(estado)

    print(f"=== Generacion {estado.generation} ===")
    print(f"Mejor score: {estado.mejor().score:.4f}   "
          f"racha sin mejora: {estado.non_improving_streak}/{PACIENCIA}")

    tabla = pd.DataFrame([
        {"origin": h.get("origin", "?"), "score": h["candidate"].score,
         **h["candidate"].params()}
        for h in hijos
    ]).sort_values("score", ascending=False)
    display(tabla)


## 4. Modo automático — correr hasta que deje de mejorar

Corre generaciones seguidas hasta que **el mejor score no mejore durante `PACIENCIA` generaciones consecutivas** (o hasta llegar a `MAX_GENERACIONES`). Ambos se configuran arriba, en la sección 1.

**Nota sobre un fix (2026-08-26):** hasta esta versión, la racha se reseteaba cuando *cualquier* hijo superaba al **peor** de los dos padres — o sea que el contador se reseteaba aunque el mejor score estuviera clavado, y el criterio de parada por estancamiento casi nunca disparaba. En una prueba real el mejor score no se movió durante 3 generaciones seguidas y la racha seguía marcando 0. Ahora compara el mejor score antes vs. después, que es lo que uno espera al decir "parar si no mejora en N generaciones".

Podés correr esta celda varias veces: cada vez retoma desde donde quedó `estado`. Para arrancar de cero, volvé a correr la celda de la sección 2.

In [ ]:
import time
_t0 = time.time()
_gen_inicial = estado.generation

if opt.deberia_parar(estado, paciencia=PACIENCIA, max_generaciones=MAX_GENERACIONES):
    print(f"Ya estaba detenido en la generacion {estado.generation} "
          f"(racha {estado.non_improving_streak}/{PACIENCIA}). "
          f"Volve a correr la seccion 2 para empezar de cero.")
else:
    while not opt.deberia_parar(estado, paciencia=PACIENCIA, max_generaciones=MAX_GENERACIONES):
        estado, hijos = opt.correr_generacion(
            estado, generar_hijos, OBJETIVO_FN,
            n_crossbred=N_CROSSBRED, n_wildcard=N_WILDCARD,
        )
        opt.guardar_checkpoint(estado)
        marca = "  <-- mejoro" if estado.non_improving_streak == 0 else ""
        print(f"gen {estado.generation:3d}   mejor score = {estado.mejor().score:10.4f}   "
              f"racha {estado.non_improving_streak}/{PACIENCIA}{marca}")

    motivo = (f"no mejoro en {PACIENCIA} generaciones seguidas"
              if estado.non_improving_streak >= PACIENCIA
              else "se alcanzo MAX_GENERACIONES")
    print()
    print(f"Detenido en la generacion {estado.generation} porque {motivo}.")
    print(f"Corrio {estado.generation - _gen_inicial} generaciones en {time.time()-_t0:.1f}s.")
    print(f"Mejor score final: {estado.mejor().score:.4f}")


## 5. Evolución del mejor score por generación

La curva de convergencia. Si termina en una meseta larga y plana, el criterio de parada hizo su trabajo.

In [ ]:
gens = [h["generation"] for h in estado.historial]
mejores = [max(h["parent_scores"]) for h in estado.historial]

plt.figure(figsize=(6, 4))
plt.plot(gens, mejores, marker="o")
plt.xlabel("Generación")
plt.ylabel("Mejor score")
plt.title("Convergencia del loop evolutivo")
plt.grid(alpha=0.3)
plt.show()


## 6. Exportar el diseño final

Registro de los valores finales, con los 18 parámetros tal cual los usa `construir_avion()`. **No** está en el formato que espera `fusion_generar_ala.py` (que sigue en el esquema viejo de 7 parámetros), así que pasarlo a Fusion requiere adaptarlo a mano por ahora.

Recordá el pendiente #2 de arriba: mientras la función objetivo siga siendo un placeholder sin restricciones físicas, **estos números no son un diseño válido**, son una prueba de que el mecanismo funciona.

In [ ]:
mejor = estado.mejor()
p = mejor.params()

salida = {
    "generation": estado.generation,
    "score": mejor.score,
    "objetivo_usado": OBJETIVO_FN.__name__,
    "_datos_de_entrada_no_optimizados": {
        "semi_envergadura_m": opt.HALF_SPAN,
        "cg_frac_mac": opt.CG_FRAC_MAC,
        "masa_total_kg": opt.MASA_TOTAL,
    },
    "flecha_deg": p["sweep_deg"],
    "cuerdas_mm": {k: 1000*p[k] for k in ["root_chord","chord_1","chord_2","chord_3","chord_4","chord_5"]},
    "le_offsets_frac_halfspan": {k: p[k] for k in ["le_offset_1","le_offset_2","le_offset_3","le_offset_4","le_offset_5"]},
    "le_offsets_mm": {k: 1000*p[k]*opt.HALF_SPAN for k in ["le_offset_1","le_offset_2","le_offset_3","le_offset_4","le_offset_5"]},
    "torsiones_deg": {f"twist_{i}": p[f"twist_{i}"] for i in range(6)},
    "elevacion_diedro_mm": {f"elev_{i}": 1000*p[f"elev_{i}"]*opt.HALF_SPAN for i in range(1,6)},
    "diedro_equivalente_deg": float(np.degrees(np.arctan2(
        p["elev_5"]*opt.HALF_SPAN, opt.HALF_SPAN))),
    "perfiles": {
        "raiz": opt.indice_a_nombre(p["perfil_raiz"]),
        "medio": opt.indice_a_nombre(p["perfil_medio"]),
        "punta": opt.indice_a_nombre(p["perfil_punta"]),
        "posicion_del_medio_frac_semienvergadura": p["perfil_pos_medio"],
        "_nota": "perfiles REALES de la base UIUC; las secciones intermedias son mezclas lineales",
    },
    "winglet": {
        "presente": bool(p["winglet_height"] >= opt.geometria.WINGLET_MIN_H),
        "largo_desarrollado_mm": 1000*p["winglet_height"],
        "cant_deg": p["winglet_cant_deg"],
        "taper": p["winglet_taper"],
        "flecha_deg": p["winglet_sweep_deg"],
        "radio_transicion_frac_largo": p.get("winglet_radio", 0.0),
        "radio_transicion_mm": 1000*p.get("winglet_radio", 0.0)*p["winglet_height"],
    },
}

ruta_salida = Path("wing_design_params.json")
ruta_salida.write_text(json.dumps(salida, indent=2))
print(f"Guardado en {ruta_salida.resolve()}")
print(json.dumps(salida, indent=2))


## 7. Visualización 3D del diseño final

Arma la geometría del mejor candidato con `construir_avion()` (la misma función que usa el objetivo para evaluar) y la dibuja en 3D. `backend="plotly"` se ve inline, interactivo (rotar/zoom con el mouse), acá mismo en el notebook.

Si preferís una **ventana aparte** (más cómoda para inspeccionar de cerca), cambiá a `backend="pyvista"` en la celda de abajo — abre un visor 3D nativo en su propia ventana.

In [ ]:
avion = opt.construir_avion(mejor.params())

# backend="plotly"  -> inline en el notebook, interactivo
# backend="pyvista" -> ventana aparte, visor 3D nativo
avion.draw(backend="plotly")


## 8. Aerodinámica y estabilidad del diseño final, y diagrama de flujo

**(a) Los números.** Desglose de resistencia (inducida vs. perfil), margen estático, y cuánto aportó cada término al score. `Cm0` tiene que ser **positivo** para que el ala pueda equilibrarse a sustentación positiva — es lo que consiguen el reflejo del perfil y el washout.

**(b) El diagrama de flujo.** AeroBuildup no produce campo de flujo, así que para dibujarlo corremos un **VLM** sobre la geometría final — una sola vez, fuera del loop. Muestra los paneles coloreados por intensidad de vórtice y las líneas de corriente. Interactivo: rotá y hacé zoom con el mouse.

⚠️ Las líneas de corriente son del **VLM, que es invíscido**: sirven para ver el patrón de flujo, los vórtices de punta y cómo se carga el ala, pero **no muestran separación ni capa límite**. Los números de resistencia de (a) sí incluyen lo viscoso; el dibujo no.

In [ ]:
aero = opt.calcular_aero(mejor.params())
p = mejor.params()

print("=== Diseno final ===")
print(f"  L/D = {aero['LD']:.2f}        CL = {aero['CL']:.4f}")
print()
print("  Resistencia:")
print(f"    CD inducido = {aero['CD_inducido']:.6f}   ({100*aero['CD_inducido']/aero['CD']:5.1f} %)")
print(f"    CD perfil   = {aero['CD_perfil']:.6f}   ({100*aero['CD_perfil']/aero['CD']:5.1f} %)")
print(f"    CD total    = {aero['CD']:.6f}")
print()
print("  Estabilidad:")
print(f"    margen estatico SM = {100*aero['SM']:.2f} %  (banda {100*obj.SM_MIN:.0f}-{100*obj.SM_MAX:.0f}%)")
print(f"    puntaje estabilidad = {aero['puntaje_estabilidad']:.3f}")
print(f"    Cm0 = {aero['Cm0']:.4f}   (>0 para poder equilibrar a CL positivo)")
print(f"    CL de equilibrio = {aero['CL_trim']:.3f}")
print()
print("  Winglet:")
if p["winglet_height"] < opt.geometria.WINGLET_MIN_H:
    print(f"    altura = {1000*p['winglet_height']:.0f} mm  ->  SIN WINGLET")
    print("    (esperable con OP_POINT en alpha=4: la inducida es ~11% del total)")
else:
    print(f"    altura = {1000*p['winglet_height']:.0f} mm   inclinacion = {p['winglet_cant_deg']:.1f} deg")
    print(f"    ahusamiento = {p['winglet_taper']:.2f}   flecha = {p['winglet_sweep_deg']:.1f} deg")
print()
print("  Aporte de cada termino al score:")
t_ld = obj.PESO_LD * aero['LD'] / obj.LD_REFERENCIA
t_es = obj.PESO_ESTABILIDAD * aero['puntaje_estabilidad']
print(f"    L/D         : {t_ld:.4f}   ({100*t_ld/(t_ld+t_es):4.1f} %)")
print(f"    estabilidad : {t_es:.4f}   ({100*t_es/(t_ld+t_es):4.1f} %)")
print(f"    TOTAL       : {t_ld+t_es:.4f}")

pd.DataFrame({
    "termino": ["inducida", "perfil (viscosa)", "TOTAL"],
    "CD": [aero["CD_inducido"], aero["CD_perfil"], aero["CD"]],
    "% del total": [100*aero["CD_inducido"]/aero["CD"], 100*aero["CD_perfil"]/aero["CD"], 100.0],
})


In [ ]:
import aerosandbox as asb

# VLM solo para VISUALIZAR -- no se usa para evaluar candidatos.
vlm_viz = asb.VortexLatticeMethod(airplane=avion, op_point=opt.OP_POINT)
res_vlm = vlm_viz.run()

print(f"VLM (solo para el dibujo): CL = {float(res_vlm['CL']):.4f}   "
      f"CD_inducido = {float(res_vlm['CD']):.6f}")
print("Recorda: este CD es SOLO inducido. El CD real esta en la celda de arriba.")

vlm_viz.draw(backend="plotly", draw_streamlines=True,
             colorbar_label="Intensidad de vortice")


## 9. Geometría en 2D

Seis vistas del ganador: planta, perfil, frontal, lateral, y las distribuciones de cuerda y torsión a lo largo de la envergadura.

Los **puntos rojos** son las **6 estaciones de control** — las que el optimizador realmente mueve. Las curvas azules se dibujan con 80 estaciones interpoladas por PCHIP, así que lo que ves es la forma suave real, no la poligonal de 6 tramos.

Sobre las escalas: planta y perfil van en escala 1:1. Las vistas frontal y lateral **no** — son franjas de 2.2 m de ancho por unos centímetros de alto, y con escala igual quedarían una raya ilegible.

In [ ]:
fig_geo = opt.plot_geometria_2d(mejor.params())
plt.show()


## 10. Performance y velocidad de pérdida

Tres curvas, todas sobre un **barrido completo de ángulo de ataque** (no el punto fijo del objetivo).

**Panel 1 — L y D contra velocidad, a α fijo.** La ley física desnuda: ambas crecen con V². No corresponde a ningún vuelo real (si mantenés α y acelerás, subís), pero muestra de dónde salen las fuerzas.

**Panel 2 — vuelo nivelado.** Acá la sustentación está fija (L = peso) y es el ángulo de ataque el que se ajusta a cada velocidad. La resistencia tiene un **mínimo**: a baja velocidad domina la inducida (mucho CL), a alta velocidad domina la de perfil (mucha presión dinámica). **Esta es la curva que sirve** para elegir velocidad de crucero y estimar autonomía.

**Panel 3 — CL contra α, con la pérdida.** NeuralFoil sí modela la caída de sustentación después de la pérdida, así que el máximo de esta curva es real y no una recta que sigue subiendo (que es lo que daría un método puramente potencial).

⚠️ **La velocidad de pérdida depende de la masa**, y la masa es hoy una **estimación declarada a mano**: `MASA_TOTAL = 3.0 kg` en `geometria.py`. No sale de ningún modelo estructural — es un supuesto razonable para un ala volante de 2.2 m con batería y payload. Cuando exista el modelo de peso (Nota 11), este número deja de ser un supuesto. Mientras tanto, si cambiás la masa, la velocidad de pérdida escala con √(masa).

In [ ]:
fig_perf, datos_vuelo = opt.plot_performance(mejor.params())
plt.show()

print(f"CL maximo         : {datos_vuelo['CL_max']:.3f}  (a alpha = {datos_vuelo['alpha_stall']:.1f} deg)")
print(f"Velocidad de perdida: {datos_vuelo['V_stall']:.2f} m/s  =  {3.6*datos_vuelo['V_stall']:.1f} km/h")
print(f"   (con masa = {opt.MASA_TOTAL} kg, S = {datos_vuelo['S']:.3f} m2)")
if datos_vuelo["stall_en_el_borde"]:
    print("  OJO: el CL maximo cayo en el borde del barrido -> ampliar alpha_max en barrido_alpha()")


## 11. Análisis de misión del diseño final

Desglose completo de lo que hace el diseño ganador en la misión pedida: **crucero a 60 km/h con 3 kg de carga útil**, en el entorno de viento de Neuquén.

Tres cosas cambiaron respecto de las secciones anteriores, y valen la aclaración:

**La masa se calcula, no se declara.** `estructura.py` dimensiona el larguero a partir de la geometría —por rigidez y por resistencia, tomando el que mande— y resuelve la circularidad (la estructura pesa según la carga, y la carga incluye a la estructura). Antes `MASA_TOTAL = 3 kg` era un supuesto puesto a mano.

**Se evalúa en la condición de crucero real**, buscando el ángulo de ataque donde la sustentación iguala al peso a 60 km/h. Esto cierra el agujero que venía desde el principio: el modelo ya no regala nada por agrandar el ala, porque un ala más pesada necesita más CL y arrastra más.

**Hay términos de viento.** El de fondo es la ráfaga vertical, que cambia el ángulo de ataque de golpe. Se aplica el **factor de alivio de ráfaga de la FAR/CS-23** (`Kg`), sin el cual la cuenta da absurdos: 3 m/s verticales a 16.7 m/s darían 10° de cambio instantáneo y ningún diseño pasaría. El avión no siente una ráfaga de canto de golpe.

⚠️ **El modelo estructural es de predimensionamiento.** Estima larguero y recubrimiento; **no ve** pandeo, torsión ni flameo (flutter). Un ala muy esbelta puede pasar este chequeo y aun así ser inviable. Los materiales y criterios (σ admisible, módulo, densidades, factor de carga) están arriba de `estructura.py` y son supuestos declarados, no medidas.

In [ ]:
print(opt.desglose_mision(mejor.params()))


## 12. Qué perfiles eligió, y de dónde salen

Ya no hay "calidad de ajuste" que medir: el optimizador **no aproxima** perfiles, los **usa**. Lo que sí vale la pena mirar es qué eligió para cada zona y cómo se ve la transición.

Recordar que las secciones **entre** dos anclajes son mezclas lineales de los dos perfiles vecinos (`blend_with_another_airfoil`), que es la práctica estándar de la industria. Una mezcla no es un perfil de catálogo, así que ahí NeuralFoil vuelve a interpolar — pero interpola *entre* dos puntos de su set de entrenamiento, no afuera, que es la diferencia de fondo con el esquema anterior.

In [ ]:
print(opt.resumen_catalogo())
print()
perf = opt.perfiles_del_diseno(mejor.params())
print(f"DISENO FINAL: raiz={perf['raiz']}  medio={perf['medio']} (en y/b={perf['pos_medio']:.2f})  punta={perf['punta']}")

# Los perfiles reales que ve el solver en cada seccion del ala
import matplotlib.pyplot as plt
av_f = opt.construir_avion(mejor.params())
n_ala = opt.N_SECCIONES_LOFT
print("Perfil por seccion:")
for i, xs in enumerate(av_f.wings[0].xsecs[:n_ala]):
    print(f"  y/b = {i/(n_ala-1):.2f}   {xs.airfoil.name}")

fig, ax = plt.subplots(figsize=(10, 3))
for frac, estilo, etq in [(0.0, "k-", "raiz"), (perf["pos_medio"], "b-", "medio"), (1.0, "g-", "punta")]:
    c = opt.perfil_en_fraccion(mejor.params(), frac).coordinates
    ax.plot(c[:, 0], c[:, 1], estilo, lw=2, label=f"{etq}: {opt.perfil_en_fraccion(mejor.params(), frac).name}")
for frac in [0.25, 0.5, 0.75]:
    c = opt.perfil_en_fraccion(mejor.params(), frac).coordinates
    ax.plot(c[:, 0], c[:, 1], "-", color="gray", lw=.7, alpha=.6)
ax.set_aspect("equal"); ax.grid(alpha=.3); ax.legend(fontsize=8)
ax.set_title("Perfiles ancla del diseno final (grises: mezclas de transicion)", fontsize=10, weight="bold")
plt.show()

## 13. Exportar la geometría a Fusion

El CAD tiene que ser **exactamente** la geometría que evaluó el solver, no una reconstrucción parecida. Por eso el reparto de tareas es tajante:

- `optimizacion_ala/exportar_cad.py` (acá, con AeroSandbox) calcula las **coordenadas 3D de cada punto de cada sección**, usando el mismo marco de referencia interno que usa el solver, y las escribe en un JSON.
- `fusion_generar_ala.py` (adentro de Fusion) **lee esos puntos y los dibuja**. No calcula nada.

La alternativa —pasarle a Fusion los parámetros y rearmar el ala con trigonometría— produce sin falta un ala levemente distinta de la analizada, y el error es silencioso. Basta con aplicar la torsión respecto de otro eje, o no reproducir el bisel a inglete que AeroSandbox mete en los quiebres.

**Dos cosas que el exportador hace y conviene saber:**

1. **Abre el borde de fuga 0.6 mm.** Los perfiles UIUC cierran en filo *exacto* (separación medida: 0.0 mm). Eso no es fabricable —en espuma o impresión 3D el borde real va a tener medio milímetro— y además es ambiguo para Fusion, que puede interpretar el contorno como spline cerrada y **redondear el pico sin avisar**. Se abre una cantidad chica y declarada: 0.12% de la cuerda de raíz, 0.75% de la de punta. Se desactiva con `espesor_te_mm=0`.
2. **Ala y winglet son dos lofts separados**, que comparten la sección de la punta. Con `winglet_radio = 0` la unión es un quiebre vivo y un loft único a través de un quiebre así sale retorcido.

Correr `verificar()` **antes** de mandar nada a Fusion: atrapa en segundos los errores que adentro del CAD recién se ven cuando el loft falla o sale trenzado (secciones no planas, orden de puntos inconsistente entre secciones, cuerdas degeneradas).

In [ ]:
from optimizacion_ala.exportar_cad import exportar_json, verificar

chequeos = verificar(mejor.params())
print("Chequeos previos al CAD:")
for k, v in chequeos.items():
    marca = "" if not k.endswith("_ok") else ("  OK" if v else "  <-- REVISAR")
    print(f"  {k:24s} {v}{marca}")

if all(v for k, v in chequeos.items() if k.endswith("_ok")):
    ruta = exportar_json(mejor.params(), "wing_geometry_cad.json", score=mejor.score)
    print(f"\nExportado: {ruta.resolve()}  ({ruta.stat().st_size/1024:.0f} KB)")
    print("Copiar ESE archivo al lado de fusion_generar_ala.py, y correr el script")
    print("desde Fusion: UTILITIES > ADD-INS > Scripts and Add-Ins > Run.")
else:
    print("\nHay chequeos en falso: no exporto. Ver arriba cual fallo.")